# K-Means Clustering of NYC Public School and MTA Subway Data

In [12]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [13]:
subway_df = pd.read_csv('../cleaned_data/MTA_Subway_Grouped_Data.csv')

subway_df.head()

,line,num_stations,station_census_tracts,station_boroughs,stop_names,mta_performance_data
0,1,38,"[285, 283, 283, 309, 303, 293, 283, 277, 269, ...","['Bx', 'Bx', 'Bx', 'M', 'M', 'M', 'M', 'M', 'M...","['Van Cortlandt Park-242 St', '238 St', '231 S...","[['2015-01-01', 76.2337806, 1], ['2015-02-01',..."
1,2,49,"[183, 159, 113, 101, 71, 21, 21, 15, 7, 5, 11,...","['M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', ...","['96 St', '72 St', 'Times Sq-42 St', '34 St-Pe...","[['2015-01-01', 47.2874494, 1], ['2015-02-01',..."
2,3,34,"[183, 159, 113, 101, 71, 21, 21, 15, 7, 5, 11,...","['M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', ...","['96 St', '72 St', 'Times Sq-42 St', '34 St-Pe...","[['2015-01-01', 68.6321563, 1], ['2015-02-01',..."
3,4,28,"[37, 35, 213, 351, 431, 419, 409, 403, 401, 23...","['Bk', 'Bk', 'Bk', 'Bk', 'Bx', 'Bx', 'Bx', 'Bx...","['Nevins St', 'Atlantic Av-Barclays Ctr', 'Fra...","[['2015-01-01', 48.4323757, 1], ['2015-02-01',..."
4,5,45,"[37, 35, 213, 319, 329, 804, 820, 826, 830, 78...","['Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk...","['Nevins St', 'Atlantic Av-Barclays Ctr', 'Fra...","[['2015-01-01', 49.3452473, 1], ['2015-02-01',..."


In [14]:
schools_df = pd.read_csv('../cleaned_data/AllBoroughs_yearly_ratings.csv')
schools_df.head(10)

,School Building Number,QR Score,Assessment Time,Location Name,Primary Address,City,Zip,Census Tract,Community District
0,K544,6,2006-04-12,NaN,NaN,NaN,NaN,NaN,NaN
1,K544,6,2007-04-28,NaN,NaN,NaN,NaN,NaN,NaN
2,K544,7,2008-03-15,NaN,NaN,NaN,NaN,NaN,NaN
3,K544,6,2010-02-10,NaN,NaN,NaN,NaN,NaN,NaN
4,K544,0,2014-12-03,NaN,NaN,NaN,NaN,NaN,NaN
5,K337,6,2006-04-26,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600.0,313.0
6,K337,1,2007-05-23,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600.0,313.0
7,K337,7,2008-05-15,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600.0,313.0
8,K337,7,2009-05-02,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600.0,313.0
9,K337,5,2012-03-28,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600.0,313.0


In [15]:
schools_df['Census Tract'] = schools_df['Census Tract'].astype(float).astype('Int64').astype(str)
schools_df.head(10)

,School Building Number,QR Score,Assessment Time,Location Name,Primary Address,City,Zip,Census Tract,Community District
0,K544,6,2006-04-12,NaN,NaN,NaN,NaN,<NA>,NaN
1,K544,6,2007-04-28,NaN,NaN,NaN,NaN,<NA>,NaN
2,K544,7,2008-03-15,NaN,NaN,NaN,NaN,<NA>,NaN
3,K544,6,2010-02-10,NaN,NaN,NaN,NaN,<NA>,NaN
4,K544,0,2014-12-03,NaN,NaN,NaN,NaN,<NA>,NaN
5,K337,6,2006-04-26,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600,313.0
6,K337,1,2007-05-23,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600,313.0
7,K337,7,2008-05-15,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600,313.0
8,K337,7,2009-05-02,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600,313.0
9,K337,5,2012-03-28,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600,313.0


# Goal:
- Use train performance as a predictor of school performance

In [28]:
schools_dim = schools_df[['Census Tract', 'Assessment Time', 'QR Score']].copy()

schools_dim = schools_dim[schools_dim['Census Tract'] != '<NA>']

schools_dim.head()

,Census Tract,Assessment Time,QR Score
5,30600,2006-04-26,6
6,30600,2007-05-23,1
7,30600,2008-05-15,7
8,30600,2009-05-02,7
9,30600,2012-03-28,5


In [32]:
subway_df.head() # 26 rows

,line,num_stations,station_census_tracts,station_boroughs,stop_names,mta_performance_data
0,1,38,"[285, 283, 283, 309, 303, 293, 283, 277, 269, ...","['Bx', 'Bx', 'Bx', 'M', 'M', 'M', 'M', 'M', 'M...","['Van Cortlandt Park-242 St', '238 St', '231 S...","[['2015-01-01', 76.2337806, 1], ['2015-02-01',..."
1,2,49,"[183, 159, 113, 101, 71, 21, 21, 15, 7, 5, 11,...","['M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', ...","['96 St', '72 St', 'Times Sq-42 St', '34 St-Pe...","[['2015-01-01', 47.2874494, 1], ['2015-02-01',..."
2,3,34,"[183, 159, 113, 101, 71, 21, 21, 15, 7, 5, 11,...","['M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', ...","['96 St', '72 St', 'Times Sq-42 St', '34 St-Pe...","[['2015-01-01', 68.6321563, 1], ['2015-02-01',..."
3,4,28,"[37, 35, 213, 351, 431, 419, 409, 403, 401, 23...","['Bk', 'Bk', 'Bk', 'Bk', 'Bx', 'Bx', 'Bx', 'Bx...","['Nevins St', 'Atlantic Av-Barclays Ctr', 'Fra...","[['2015-01-01', 48.4323757, 1], ['2015-02-01',..."
4,5,45,"[37, 35, 213, 319, 329, 804, 820, 826, 830, 78...","['Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk...","['Nevins St', 'Atlantic Av-Barclays Ctr', 'Fra...","[['2015-01-01', 49.3452473, 1], ['2015-02-01',..."
